# 2. Propagation methods

LUMOS propagates a field sampled on any grid to any other grid. Several methods exist (Fresnel transfer
function and impulse response, one-step Fresnel with a matrix Fourier transform, band-limited angular
spectrum, direct Rayleigh-Sommerfeld); the **planner** chooses the most efficient one that respects the
sampling criteria, and logs its choice.

In [ ]:
import numpy as np
from astropy import units as u

import lumos
from lumos import propagation
from lumos.fields import Grid, Wavefront
from lumos.plotting import style
from lumos.processing import radial_profile

style.apply()

In [ ]:
grid = Grid(256, dx=20 * u.um)
beam = Wavefront.gaussian(grid, 633 * u.nm, waist=0.5 * u.mm, power=1 * u.mW)
z = 1 * u.m
propagation.plan(beam, z)

## Every method against the analytic Gaussian beam

The beam radius grows as $w(z) = w_0 \sqrt{1 + (z / z_R)^2}$ with the Rayleigh range
$z_R = \pi w_0^2 / \lambda$ (Siegman, *Lasers*, §17.1).

In [ ]:
methods = ["fresnel_tf", "fresnel_ir", "fresnel_mft", "asm"]
outputs = {name: propagation.propagate(beam, z, method=name) for name in methods}
z_r = (np.pi * (0.5 * u.mm) ** 2 / (633 * u.nm)).to(u.m)
w_z = 0.5 * u.mm * np.sqrt(1 + (z / z_r).decompose() ** 2)

fig, ax = style.figure("single")
for name, field in outputs.items():
    r, profile = radial_profile(field.intensity, field.grid)
    ax.plot(r.to_value(u.mm), profile, label=name)
peak = float(outputs["asm"].intensity.max())
r = np.linspace(0, 2.5, 200) * u.mm
analytic = peak * np.exp(-2 * (r / w_z).decompose().value ** 2)
ax.plot(r.value, analytic, "k--", label="analytic")
ax.set_xlabel(style.axis_label("r", u.mm))
ax.set_ylabel(style.axis_label("intensity", outputs["asm"].intensity_unit))
ax.legend();

In [ ]:
{name: field.power.to(u.mW) for name, field in outputs.items()}

## The same propagation inside a context

The preset `propagation_methods.lumos` lets the planner choose: its choice is in the journal of the context.

In [ ]:
ctx = lumos.io.load("presets/propagation_methods.lumos")
result = ctx.run(ctx.meta["outputs"], archive="propagation-methods")
[entry.message for entry in ctx.journal.entries("log") if "method" in entry.message]

In [ ]:
result.archive.save_figure(fig, "methods", data={"r": r, "analytic": analytic})
result["meter.out"].data